# ICU Mortality Prediction - Modeling

## Author
Siva Ram

## Overview
This notebook performs predictive modeling on the final curated ICU dataset. The goal is to predict hospital mortality using clinical features collected during the first 24 hours of ICU admission.

We evaluate three feature configurations:
1. APACHE-only features
2. Non-APACHE-only features
3. Combined feature set

For each configuration, we train:
- Logistic Regression (baseline model)
- Random Forest
- Gradient Boosting

Model performance is evaluated using:
- ROC-AUC
- Precision
- Recall
- F1-score

In [1]:
# Import required libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

# Display settings
pd.set_option("display.max_columns", None)

## 1. Load Final Dataset

The final dataset was created during the EDA and data preparation phase. It contains:
- 8 APACHE features
-  Corresponding non-APACHE features
- 1 target variable: `hospital_death`

In [2]:
# Load the final curated dataset
df = pd.read_csv("/content/final_16_features_plus_target.csv")

TARGET = "hospital_death"

# Separate predictors and target
X = df.drop(columns=[TARGET])
y = df[TARGET]

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (91713, 13)


,bun_apache,map_apache,heart_rate_apache,apache_3j_diagnosis,creatinine_apache,ventilated_apache,resprate_apache,temp_apache,d1_bun_max,d1_creatinine_max,d1_resprate_max,d1_temp_max,hospital_death
0,31.0,40.0,118.0,502.01,2.51,0.0,36.0,39.3,31.0,2.51,34.0,39.9,0
1,9.0,46.0,120.0,203.01,0.56,1.0,33.0,35.1,11.0,0.71,32.0,36.3,0
2,NaN,68.0,102.0,703.03,NaN,0.0,37.0,36.7,NaN,NaN,21.0,37.0,0
3,NaN,60.0,114.0,1206.03,NaN,1.0,4.0,34.8,NaN,NaN,23.0,38.0,0
4,NaN,103.0,60.0,601.01,NaN,0.0,16.0,36.7,NaN,NaN,18.0,37.2,0


## 2. Identify Feature Groups

To compare the predictive value of APACHE and non-APACHE features, the predictors are divided into two groups:
- APACHE features
- Non-APACHE features

In [5]:
# Identify APACHE and non-APACHE columns
apache_features = [c for c in X.columns if "apache" in c.lower()]
non_apache_features = [c for c in X.columns if "apache" not in c.lower()]

print("APACHE features:")
print(apache_features)

print("\nNon-APACHE features:")
print(non_apache_features)

APACHE features:
['bun_apache', 'map_apache', 'heart_rate_apache', 'apache_3j_diagnosis', 'creatinine_apache', 'ventilated_apache', 'resprate_apache', 'temp_apache']

Non-APACHE features:
['d1_bun_max', 'd1_creatinine_max', 'd1_resprate_max', 'd1_temp_max']


## 3. Create Feature Sets

Three feature configurations are evaluated:
1. APACHE-only
2. Non-APACHE-only
3. Combined

In [6]:
# Define feature sets for comparison
feature_sets = {
    "APACHE only": apache_features,
    "Non-APACHE only": non_apache_features,
    "Combined": apache_features + non_apache_features
}

## 4. Train-Test Split

The data is split into training and testing sets using stratified sampling to preserve the class distribution of the mortality outcome.

In [7]:
# Create train-test split
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set shape:", X_train_full.shape)
print("Testing set shape:", X_test_full.shape)

Training set shape: (73370, 12)
Testing set shape: (18343, 12)


## 5. Define Modeling Function

The following function trains and evaluates three models for a given feature set:
- Logistic Regression
- Random Forest
- Gradient Boosting

In [8]:
def evaluate_models(feature_list, feature_set_name):
    """
    Train and evaluate Logistic Regression, Random Forest,
    and Gradient Boosting on the specified feature set.
    """

    # Subset the train and test data for the selected features
    X_train = X_train_full[feature_list]
    X_test = X_test_full[feature_list]

    # Preprocessing pipeline:
    # - median imputation for missing values
    # - standardization for numeric features
    preprocess = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), feature_list)
        ]
    )

    results = []

    # =========================
    # 1. Logistic Regression
    # =========================
    log_model = Pipeline([
        ("prep", preprocess),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    log_model.fit(X_train, y_train)
    y_pred_lr = log_model.predict(X_test)
    y_prob_lr = log_model.predict_proba(X_test)[:, 1]

    results.append({
        "Feature Set": feature_set_name,
        "Model": "Logistic Regression",
        "ROC-AUC": roc_auc_score(y_test, y_prob_lr),
        "Precision": precision_score(y_test, y_pred_lr, zero_division=0),
        "Recall": recall_score(y_test, y_pred_lr, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred_lr, zero_division=0)
    })

    # =========================
    # 2. Random Forest
    # =========================
    rf_model = Pipeline([
        ("prep", preprocess),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced"
        ))
    ])

    rf_model.fit(X_train, y_train)
    y_pred_rf = rf_model.predict(X_test)
    y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

    results.append({
        "Feature Set": feature_set_name,
        "Model": "Random Forest",
        "ROC-AUC": roc_auc_score(y_test, y_prob_rf),
        "Precision": precision_score(y_test, y_pred_rf, zero_division=0),
        "Recall": recall_score(y_test, y_pred_rf, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred_rf, zero_division=0)
    })

    # =========================
    # 3. Gradient Boosting
    # =========================
    gb_model = Pipeline([
        ("prep", preprocess),
        ("model", GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42
        ))
    ])

    gb_model.fit(X_train, y_train)
    y_pred_gb = gb_model.predict(X_test)
    y_prob_gb = gb_model.predict_proba(X_test)[:, 1]

    results.append({
        "Feature Set": feature_set_name,
        "Model": "Gradient Boosting",
        "ROC-AUC": roc_auc_score(y_test, y_prob_gb),
        "Precision": precision_score(y_test, y_pred_gb, zero_division=0),
        "Recall": recall_score(y_test, y_pred_gb, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred_gb, zero_division=0)
    })

    return results

## 6. Run Experiments

Each of the three models is trained and evaluated on each feature configuration.

In [9]:
# Run all experiments
all_results = []

for feature_set_name, feature_list in feature_sets.items():
    results = evaluate_models(feature_list, feature_set_name)
    all_results.extend(results)

# Convert results to DataFrame
results_df = pd.DataFrame(all_results)

# Round values for easier reading
results_df[["ROC-AUC", "Precision", "Recall", "F1 Score"]] = results_df[
    ["ROC-AUC", "Precision", "Recall", "F1 Score"]
].round(4)

results_df

,Feature Set,Model,ROC-AUC,Precision,Recall,F1 Score
0,APACHE only,Logistic Regression,0.8055,0.2146,0.7214,0.3308
1,APACHE only,Random Forest,0.8438,0.5283,0.1592,0.2447
2,APACHE only,Gradient Boosting,0.8576,0.6957,0.2123,0.3253
3,Non-APACHE only,Logistic Regression,0.7020,0.1627,0.5287,0.2488
4,Non-APACHE only,Random Forest,0.6977,0.2776,0.0954,0.1420
5,Non-APACHE only,Gradient Boosting,0.7644,0.7161,0.0701,0.1277
6,Combined,Logistic Regression,0.8096,0.2165,0.7378,0.3348
7,Combined,Random Forest,0.8510,0.7205,0.1466,0.2436
8,Combined,Gradient Boosting,0.8631,0.6990,0.2186,0.3330


## 7. Compare Results

The table above compares the performance of Logistic Regression, Random Forest, and Gradient Boosting across APACHE-only, non-APACHE-only, and combined feature sets.

In [10]:
# Sort results by ROC-AUC for easier comparison
results_df.sort_values(by="ROC-AUC", ascending=False)

,Feature Set,Model,ROC-AUC,Precision,Recall,F1 Score
8,Combined,Gradient Boosting,0.8631,0.6990,0.2186,0.3330
2,APACHE only,Gradient Boosting,0.8576,0.6957,0.2123,0.3253
7,Combined,Random Forest,0.8510,0.7205,0.1466,0.2436
1,APACHE only,Random Forest,0.8438,0.5283,0.1592,0.2447
6,Combined,Logistic Regression,0.8096,0.2165,0.7378,0.3348
0,APACHE only,Logistic Regression,0.8055,0.2146,0.7214,0.3308
5,Non-APACHE only,Gradient Boosting,0.7644,0.7161,0.0701,0.1277
3,Non-APACHE only,Logistic Regression,0.7020,0.1627,0.5287,0.2488
4,Non-APACHE only,Random Forest,0.6977,0.2776,0.0954,0.1420


## 8. Interpretation of Results

The modeling results show several important patterns:

1. **Combined features performed best overall**, indicating that APACHE and non-APACHE variables provide complementary information.
2. **Gradient Boosting with the combined feature set achieved the highest ROC-AUC**, making it the strongest model for overall discrimination.
3. **Logistic Regression with the combined feature set achieved the highest recall and F1-score**, making it useful when identifying as many mortality cases as possible is important.
4. **APACHE-only models consistently outperformed Non-APACHE-only models**, suggesting that APACHE-derived severity features are especially informative for this prediction task.
5. **Non-APACHE-only models had the weakest performance overall**, although Gradient Boosting still performed reasonably in ROC-AUC terms.

These results suggest that:
- APACHE features carry strong predictive signal
- non-APACHE features add useful complementary information
- combining both gives the best overall performance

## 9. Identify the Best Model

The final model is selected based primarily on ROC-AUC, since ROC-AUC is a robust metric for imbalanced binary classification problems.

Based on this criterion, the best model is expected to be **Gradient Boosting with the combined feature set**.

In [11]:
# Select the best-performing model row based on ROC-AUC
best_model_row = results_df.sort_values(by="ROC-AUC", ascending=False).iloc[0]
best_model_row

,8
Feature Set,Combined
Model,Gradient Boosting
ROC-AUC,0.8631
Precision,0.699
Recall,0.2186
F1 Score,0.333


## 10. Train Final Selected Model

Since Gradient Boosting with the combined feature set achieved the highest ROC-AUC, it is retrained here on the combined training data so it can be saved for later use in deployment.

In [12]:
# Use combined feature set for final model
final_features = feature_sets["Combined"]

X_train = X_train_full[final_features]
X_test = X_test_full[final_features]

# Preprocessing pipeline
final_preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), final_features)
    ]
)

# Final selected model: Gradient Boosting on combined features
final_model = Pipeline([
    ("prep", final_preprocess),
    ("model", GradientBoostingClassifier(
        n_estimators=50,
        learning_rate=0.1,
        max_depth=2,
        random_state=42
    ))
])

# Train final model
final_model.fit(X_train, y_train)

# Evaluate final model
y_pred_final = final_model.predict(X_test)
y_prob_final = final_model.predict_proba(X_test)[:, 1]

print("Final Model: Gradient Boosting + Combined Features")
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_final), 4))
print("Precision:", round(precision_score(y_test, y_pred_final, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred_final, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_test, y_pred_final, zero_division=0), 4))

Final Model: Gradient Boosting + Combined Features
ROC-AUC: 0.8485
Precision: 0.7568
Recall: 0.1396
F1 Score: 0.2357


## 11. Save Final Model

The final trained pipeline is saved so it can later be used in the Streamlit application.

In [14]:
# Save final trained model pipeline
import joblib

joblib.dump(final_model, "/content/final_gradient_boosting_model.pkl")

print("Final model saved to /content/final_gradient_boosting_model.pkl")

Final model saved to /content/final_gradient_boosting_model.pkl


## 12. Conclusion

This notebook compared Logistic Regression, Random Forest, and Gradient Boosting across APACHE-only, non-APACHE-only, and combined feature sets for ICU mortality prediction.

### Main findings
- The **combined feature set** performed best overall.
- **Gradient Boosting + Combined** achieved the highest ROC-AUC (**0.8631**), making it the best overall classifier.
- **Logistic Regression + Combined** achieved the highest recall (**0.7378**) and the highest F1-score (**0.3348**), making it a strong interpretable baseline.
- APACHE-only models were stronger than Non-APACHE-only models, showing that APACHE-derived severity variables are highly informative.

Overall, the results suggest that combining APACHE and non-APACHE features improves prediction, and Gradient Boosting is the best final model for deployment.

## 13. Next Steps

The next steps are:
- use the saved final model in the Streamlit application
- summarize the modeling results in the project report
- include the best model and evaluation metrics in the presentation